#  HTGS8F Processing

- Whole genome sequence data; Illumina and Nanopore.
- Shipped and submitted to Plasmidsaurus ???.
- Data available on: ???.
- Started procecessing on: 11-25-25 on Poplar.

## Previous analysis
Data was previously downloaded to nspahr laptop (before direct download via Plasmidsaurus REST API was implemented) and a preliminary analysis was done there. Short read data was filtered/trimmed, and mutations were called against ADP1 and ACN2586. However, only the received and trimmed reads were uploaded to poplar. The samples in this order are of different lineages and should not all be compared to the same parent strain.

## Present analysis
I uploaded the data to poplar and initiated QA/QC (steps 1 and 2 in this notebook. These cells may not be executable any longer because the pipeline code has been continuously improved since starting the analysis). Later, I started mutation calling for a subset of the samples, i.e. those strains, that were of current interest because they were used in a robotic ALE experiment. This notebook can be used to run different batches of the samples against different references (at different time points.)

## Steps
1. Data organization, file renaming
    1.  Get data onto server.
    2.  Compress library folder (right-click, compress)
    3.  scp into my jupyter home: scp -prJ nspahr@logins.cels.anl.gov /Users/nataschaspahr/data/seq_data/Plasmidsaurus_10-06-2025_P4CYGL.zip nspahr@poplar.cels.anl.gov:/scratch1/fliu/hub_scratch/nspahr/
2. Short reads: QA/QC
3. Notebook set-up for current version of AISynbioPipeline code.
4. Creating seqsamples and cross-checking in LIMS.
5. Breseq: Batch 1 (Isolates of 5 different ACN3500 progeny strains)

## 1. Data organization

In [1]:
## Running on poplar
## Make sure running in appropriate kernel

import sys, os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

# Must add ai_synbio repo to PATH
sys.path.append('/home/nspahr/code/ai_synbio_data_processing/')

In [2]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

In [3]:
# Define AI-Synbio project data directories

LIB_DIR = '/storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/'
library = 'Plasmidsaurus_9-13-2025_HTGS8F'
lib_path = os.path.join(LIB_DIR, library)
shortlib = os.path.join(lib_path, os.path.basename(lib_path) + '_Illumina')
longlib = os.path.join(lib_path, os.path.basename(lib_path) + '_Nanopore')

In [4]:
# Define my Jupyter Hub home directories for viewing intermittent results

home_lib_path = os.path.join('/storage/nspahr/', 'lib_analysis', library)
os.makedirs(home_lib_path, exist_ok=True)

## 2. Short reads: QA/QC

In [5]:
short_received = os.path.join(shortlib, 'received')

In [10]:
import os
from binfo_utils import create_manifest

manifest = create_manifest(short_received, platform='plasmidsaurus_hybrid')
manifest

,sample_name,fwd_fastq,rvs_fastq
23,ANLstock.ACN2853.pyruvate.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
8,ANLstock.ACN2853.pyruvate.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
20,ANLstock.ACN2853.pyruvate.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
24,ANLstock.ACN2853_T_dgoA-Best.pyruvate.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
25,ANLstock.ACN2853_T_dgoA-Best.pyruvate.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
26,ANLstock.ACN2853_T_dgoA-Best.pyruvate.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
19,ANLstock.ACN3210.pyruvate.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
22,ANLstock.ACN3210.pyruvate.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
21,ANLstock.ACN3210.pyruvate.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
16,ANLstock.ACN3560.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


### QC of received

In [11]:
from read_qc import run_fastqc
from binfo_utils import create_manifest

manifest = create_manifest(short_received, platform='plasmidsaurus_hybrid')

[run_fastqc(f) for f in manifest['fwd_fastq']]
[run_fastqc(f) for f in manifest['rvs_fastq']]

Running FastQC on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
application/gzip


Started analysis of ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 5% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 10% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 15% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 20% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 25% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 30% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 35% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 40% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 45% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 50% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 55% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Approx 60% complete for ANLstock.ACN2853.pyruvate.colony1

Analysis complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz
Running FastQC on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
application/gzip


Started analysis of ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 5% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 10% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 15% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 20% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 25% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 30% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 35% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 40% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 45% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 50% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 55% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Approx 60% complete for ANLstock.ACN2853.pyruvate.colony2

Analysis complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz
Running FastQC on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ACN2853.pyruvate.colony3_illumina_R1.fastq.gz
application/gzip


...

Started analysis of ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 5% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 10% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 15% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 20% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 25% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 30% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 35% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 40% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 45% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 50% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 55% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 60% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 65% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Approx 70% complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz


Analysis complete for ANLstock.ADP1.colony2_illumina_R2.fastq.gz
Running FastQC on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ADP1.colony3_illumina_R2.fastq.gz
application/gzip


Started analysis of ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 5% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 10% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 15% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 20% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 25% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 30% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 35% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 40% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 45% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 50% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 55% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 60% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 65% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz
Approx 70% complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz


Analysis complete for ANLstock.ADP1.colony3_illumina_R2.fastq.gz


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [12]:
from read_qc import run_multiqc
import shutil

multiqc_report = run_multiqc(short_received)
shutil.copy(multiqc_report, home_lib_path)


/// ]8;id=442942;https://multiqc.info\MultiQC]8;;\ v1.32

       file_search | Search path: /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received


        searching | ████████████████████████████████████████ 100% 162/162                                                   html

            fastqc | Found 54 reports
     write_results | Data        : multiqc_data
     write_results | Report      : multiqc_report.html
           multiqc | MultiQC complete


'/storage/nspahr/lib_analysis/Plasmidsaurus_9-13-2025_HTGS8F/multiqc_report.html'

### Trimming with fastp

In [6]:
import subprocess
from binfo_utils import create_manifest
from read_qc import run_fastp

short_trimmed = os.path.join(shortlib, 'trimmed')
os.makedirs(short_trimmed, exist_ok=True)

manifest = create_manifest(short_received, platform='plasmidsaurus_hybrid')

for index, row in manifest.iterrows():
    read1_in_path = row['fwd_fastq']
    read2_in_path = row['rvs_fastq']
    read1_in_basename = os.path.basename(read1_in_path)
    read2_in_basename = os.path.basename(read2_in_path)
    read1_out_path = os.path.join(short_trimmed, read1_in_basename.replace('.fastq.gz', '_trimmed.fastq.gz'))
    read2_out_path = os.path.join(short_trimmed, read2_in_basename.replace('.fastq.gz', '_trimmed.fastq.gz'))
    run_fastp(read1_in_path, read2_in_path, read1_out_path, read2_out_path, threads=16, polyG=6)

Running fastp on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ACN2853.pyruvate.colony1_illumina_R1.fastq.gz and /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ACN2853.pyruvate.colony1_illumina_R2.fastq.gz


Read1 before filtering:
total reads: 23250157
total bases: 3510773707
Q20 bases: 3490850780(99.4325%)
Q30 bases: 3420290284(97.4227%)
Q40 bases: 3420290284(97.4227%)

Read2 before filtering:
total reads: 23250157
total bases: 3510773707
Q20 bases: 3456335737(98.4494%)
Q30 bases: 3308704727(94.2443%)
Q40 bases: 3308704727(94.2443%)

Read1 after filtering:
total reads: 23045981
total bases: 3462024147
Q20 bases: 3447959457(99.5937%)
Q30 bases: 3385120403(97.7786%)
Q40 bases: 3385120403(97.7786%)

Read2 after filtering:
total reads: 23045981
total bases: 3462024147
Q20 bases: 3422953164(98.8714%)
Q30 bases: 3287496311(94.9588%)
Q40 bases: 3287496311(94.9588%)

Filtering result:
reads passed filter: 46091962
reads failed due to low quality: 398754
reads failed due to too many N: 9598
reads failed due to too short: 0
reads with adapter trimmed: 768174
bases trimmed due to adapters: 35882350

Duplication rate: 28.8134%

Insert size peak (evaluated by paired-end reads): 271

JSON report: /sto

Running fastp on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ACN2853.pyruvate.colony2_illumina_R1.fastq.gz and /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ACN2853.pyruvate.colony2_illumina_R2.fastq.gz


Read1 before filtering:
total reads: 28188720
total bases: 4256496720
Q20 bases: 4232558285(99.4376%)
Q30 bases: 4143831942(97.3531%)
Q40 bases: 4143831942(97.3531%)

Read2 before filtering:
total reads: 28188720
total bases: 4256496720
Q20 bases: 4189872923(98.4348%)
Q30 bases: 4009534451(94.198%)
Q40 bases: 4009534451(94.198%)

Read1 after filtering:
total reads: 27934938
total bases: 4197294851
Q20 bases: 4179707067(99.581%)
Q30 bases: 4099794846(97.6771%)
Q40 bases: 4099794846(97.6771%)

Read2 after filtering:
total reads: 27934938
total bases: 4197294851
Q20 bases: 4150049843(98.8744%)
Q30 bases: 3984713140(94.9353%)
Q40 bases: 3984713140(94.9353%)

Filtering result:
reads passed filter: 55869876
reads failed due to low quality: 495916
reads failed due to too many N: 11648
reads failed due to too short: 0
reads with adapter trimmed: 887140
bases trimmed due to adapters: 41806374

Duplication rate: 27.1442%

Insert size peak (evaluated by paired-end reads): 271

JSON report: /stora

Running fastp on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ACN2853.pyruvate.colony3_illumina_R1.fastq.gz and /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ACN2853.pyruvate.colony3_illumina_R2.fastq.gz


...

Read1 before filtering:
total reads: 18213769
total bases: 2750279119
Q20 bases: 2732843031(99.366%)
Q30 bases: 2668137761(97.0133%)
Q40 bases: 2668137761(97.0133%)

Read2 before filtering:
total reads: 18213769
total bases: 2750279119
Q20 bases: 2715369793(98.7307%)
Q30 bases: 2614941749(95.0791%)
Q40 bases: 2614941749(95.0791%)

Read1 after filtering:
total reads: 18082706
total bases: 2716555217
Q20 bases: 2703458995(99.5179%)
Q30 bases: 2643817358(97.3224%)
Q40 bases: 2643817358(97.3224%)

Read2 after filtering:
total reads: 18082706
total bases: 2716555217
Q20 bases: 2691302363(99.0704%)
Q30 bases: 2598363443(95.6492%)
Q40 bases: 2598363443(95.6492%)

Filtering result:
reads passed filter: 36165412
reads failed due to low quality: 254504
reads failed due to too many N: 7622
reads failed due to too short: 0
reads with adapter trimmed: 606552
bases trimmed due to adapters: 27892168

Duplication rate: 24.0088%

Insert size peak (evaluated by paired-end reads): 271

JSON report: /stor

Running fastp on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ADP1.colony2_illumina_R1.fastq.gz and /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ADP1.colony2_illumina_R2.fastq.gz


Read1 before filtering:
total reads: 16118349
total bases: 2433870699
Q20 bases: 2420549056(99.4527%)
Q30 bases: 2369928469(97.3728%)
Q40 bases: 2369928469(97.3728%)

Read2 before filtering:
total reads: 16118349
total bases: 2433870699
Q20 bases: 2397199171(98.4933%)
Q30 bases: 2295606068(94.3191%)
Q40 bases: 2295606068(94.3191%)

Read1 after filtering:
total reads: 15987744
total bases: 2405620087
Q20 bases: 2395397092(99.575%)
Q30 bases: 2349054721(97.6486%)
Q40 bases: 2349054721(97.6486%)

Read2 after filtering:
total reads: 15987744
total bases: 2405620087
Q20 bases: 2378681246(98.8802%)
Q30 bases: 2284716432(94.9741%)
Q40 bases: 2284716432(94.9741%)

Filtering result:
reads passed filter: 31975488
reads failed due to low quality: 254456
reads failed due to too many N: 6754
reads failed due to too short: 0
reads with adapter trimmed: 375180
bases trimmed due to adapters: 17074030

Duplication rate: 23.7301%

Insert size peak (evaluated by paired-end reads): 268

JSON report: /stor

Running fastp on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ADP1.colony3_illumina_R1.fastq.gz and /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/received/ANLstock.ADP1.colony3_illumina_R2.fastq.gz


Read1 before filtering:
total reads: 22375952
total bases: 3378768752
Q20 bases: 3357146716(99.3601%)
Q30 bases: 3276341421(96.9685%)
Q40 bases: 3276341421(96.9685%)

Read2 before filtering:
total reads: 22375952
total bases: 3378768752
Q20 bases: 3329438495(98.54%)
Q30 bases: 3193037100(94.503%)
Q40 bases: 3193037100(94.503%)

Read1 after filtering:
total reads: 22182662
total bases: 3333517891
Q20 bases: 3317272318(99.5127%)
Q30 bases: 3243316063(97.2941%)
Q40 bases: 3243316063(97.2941%)

Read2 after filtering:
total reads: 22182662
total bases: 3333517891
Q20 bases: 3298568673(98.9516%)
Q30 bases: 3173299928(95.1937%)
Q40 bases: 3173299928(95.1937%)

Filtering result:
reads passed filter: 44365324
reads failed due to low quality: 377758
reads failed due to too many N: 8822
reads failed due to too short: 0
reads with adapter trimmed: 687330
bases trimmed due to adapters: 32160244

Duplication rate: 25.3608%

Insert size peak (evaluated by paired-end reads): 271

JSON report: /storage

### QC of fastp-trimmed

In [7]:
from read_qc import run_fastqc
from binfo_utils import create_manifest

manifest = create_manifest(short_trimmed, platform='plasmidsaurus_hybrid')

[run_fastqc(f) for f in manifest['fwd_fastq']]
[run_fastqc(f) for f in manifest['rvs_fastq']]

Running FastQC on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/trimmed/ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
application/gzip


Started analysis of ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 5% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 10% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 15% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 20% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 25% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 30% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 35% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 40% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 45% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 50% complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Approx 55% complete for ANLstock.ACN2853.pyruvat

Analysis complete for ANLstock.ACN2853.pyruvate.colony1_illumina_R1_trimmed.fastq.gz
Running FastQC on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/trimmed/ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
application/gzip


Started analysis of ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 5% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 10% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 15% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 20% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 25% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 30% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 35% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 40% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 45% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 50% complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Approx 55% complete for ANLstock.ACN2853.pyruvat

Analysis complete for ANLstock.ACN2853.pyruvate.colony2_illumina_R1_trimmed.fastq.gz
Running FastQC on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/trimmed/ANLstock.ACN2853.pyruvate.colony3_illumina_R1_trimmed.fastq.gz
application/gzip


...

Started analysis of ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 5% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 10% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 15% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 20% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 25% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 30% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 35% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 40% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 45% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 50% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 55% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 60% complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Approx 65% complete for ANLsto

Analysis complete for ANLstock.ADP1.colony2_illumina_R2_trimmed.fastq.gz
Running FastQC on /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/trimmed/ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
application/gzip


Started analysis of ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 5% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 10% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 15% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 20% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 25% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 30% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 35% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 40% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 45% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 50% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 55% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 60% complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz
Approx 65% complete for ANLsto

Analysis complete for ANLstock.ADP1.colony3_illumina_R2_trimmed.fastq.gz


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [8]:
from read_qc import run_multiqc
import shutil

multiqc_report = run_multiqc(short_trimmed)
shutil.copy(multiqc_report, os.path.join(home_lib_path, 'short_trimmed_multiqc_report.html'))


/// ]8;id=262134;https://multiqc.info\MultiQC]8;;\ v1.32

       file_search | Search path: /storage/synbio/ai_synbio_data/experimental_data/sequencing_libraries/Plasmidsaurus_9-13-2025_HTGS8F/Plasmidsaurus_9-13-2025_HTGS8F_Illumina/trimmed


        searching | ████████████████████████████████████████ 100% 216/216                                                   html

             fastp | Found 27 reports
            fastqc | Found 54 reports
     write_results | Data        : multiqc_data
     write_results | Report      : multiqc_report.html
           multiqc | MultiQC complete


'/storage/nspahr/lib_analysis/Plasmidsaurus_9-13-2025_HTGS8F/short_trimmed_multiqc_report.html'

**Results**
FastQC is terribly slow, but reports duplication levels and specifically indicates adapter sequences left after trimming. Not sure how to assess this without fastQC.

## 3. Notebook set-up

In [ ]:
## Make sure running in aisynbio_env

In [1]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

In [4]:
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders

list_seqorders()

['Plasmidsaurus_9-13-2025_HTGS8F',
 'Plasmidsaurus_10-06-2025_P4CYGL',
 'ANL_ALE1b',
 'mock_SeqOrder_12-04-25',
 'Plasmidsaurus_2025-12-22_MQRBN8',
 'Plasmidsaurus_2025-09-30_P4CYGL',
 'Plasmidsaurus_2026-01-13_S2SVV5']

In [5]:
# Create seqorder name, make reception folder and seqorder analysis folder in nspahr homedir

from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample
import os

item_code = 'HTGS8F'
seqorder_name = 'Plasmidsaurus_9-13-2025_HTGS8F'

home_dir = '/storage/nspahr/lib_analysis/' + seqorder_name
breseq_ACN3500_dir = home_dir + '/breseq_analysis_ACN3500'
os.makedirs(breseq_ACN3500_dir, exist_ok=True)

In [6]:
# Define seqorder

seqorder = SeqOrder(seqorder_name)
short = Library(seqorder, 'Illumina')

## 4. Cross-checking this Plasmidsaurus order seqsamples in LIMS and creating SeqSamples.

In [7]:
short_manifest = short.create_manifest('received')
short_manifest

,sample_name,R1,R2
0,ANLstock.ACN2853.pyruvate.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,ANLstock.ACN2853.pyruvate.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,ANLstock.ACN2853.pyruvate.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,ANLstock.ACN2853_T_dgoA-Best.pyruvate.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,ANLstock.ACN2853_T_dgoA-Best.pyruvate.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
5,ANLstock.ACN2853_T_dgoA-Best.pyruvate.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
6,ANLstock.ACN3210.pyruvate.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
7,ANLstock.ACN3210.pyruvate.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
8,ANLstock.ACN3210.pyruvate.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
9,ANLstock.ACN3560.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [8]:
# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)

%run util_simple.py

✓ LIMS API loaded successfully


In [9]:
# Run a manual LIMS mirror db sync

from aisynbiopipeline.limsapi.sync import sync_all_sheets

sync_all_sheets()

2026-02-11 04:20:32,372 - lims_sync - INFO - Starting sync operation
2026-02-11 04:20:32,373 - lims_sync - INFO - Connecting to Google Sheets
2026-02-11 04:20:33,147 - lims_sync - INFO - Connecting to database
2026-02-11 04:20:33,455 - lims_sync - INFO - Found 15 worksheets: Experiments, Strains, Conditions, Samples, Measurements, Genes, Measurement_types, DNA_constructs, Primers, dgoA_alleles_new, dgoA_alleles_old, robotic_mt_samples, Wells, Strain_stocks_ANL, Plasmid_stocks_ANL
2026-02-11 04:20:33,456 - lims_sync - INFO - Syncing worksheet: Experiments
2026-02-11 04:20:34,746 - lims_sync - INFO - Retrieved 11 rows from Experiments
2026-02-11 04:20:34,879 - lims_sync - INFO - Inserted 3, updated 0 rows in Experiments
2026-02-11 04:20:34,893 - lims_sync - INFO - Marked 1 rows as deleted in Experiments
2026-02-11 04:20:39,894 - lims_sync - INFO - Syncing worksheet: Strains
2026-02-11 04:20:41,039 - lims_sync - INFO - Retrieved 1020 rows from Strains
2026-02-11 04:20:41,198 - lims_sync -

{'start_time': '2026-02-11T04:20:32.372872',
 'end_time': '2026-02-11T04:22:18.509420',
 'success': True,
 'tables_synced': 15,
 'total_rows_inserted': 672,
 'total_rows_updated': 0,
 'total_rows_deleted': 48,
 'errors': []}

In [12]:
# Cross-checking seq sample measurement names

thisExpLIMSseqsamples_short = query_lims(
    'Measurements',
    filters={'Experiment': 'strain_stocks', 'Type': 'Short_DNA_reads'}
)['Name'].to_list()

# thisExpLIMSseqsamples_long = query_lims(
#     'Measurements',
#     filters={'Experiment': 'strain_stocks', 'Type': 'Long_DNA_reads'}
# )['Name'].to_list()

print(f"Are all short Plasmidsaurus seqsamples from order {item_code} in the LIMS?")
print(all([x in thisExpLIMSseqsamples_short for x in short_manifest['sample_name'].to_list()]))
print(f"Are all short LIMS seqsamples with selected filters in Plasmidsaurus order {item_code}?")
print(all([x in short_manifest['sample_name'].to_list() for x in thisExpLIMSseqsamples_short]))

# long_manifest = long.create_manifest('received')
# print(f"Are all Plasmidsaurus long seqsamples from order {item_code} in the LIMS?")
# print(all([x in thisExpLIMSseqsamples_long for x in long_manifest['sample_name'].to_list()]))
# print(f"Are all LIMS long seqsamples with selected filters in Plasmidsaurus order {item_code}?")
# print(all([x in long_manifest['sample_name'].to_list() for x in thisExpLIMSseqsamples_long]))

Are all short Plasmidsaurus seqsamples from order HTGS8F in the LIMS?
False
Are all short LIMS seqsamples with selected filters in Plasmidsaurus order HTGS8F?
False


NameError: name 'long' is not defined

## 5. Breseq: Batch 1 (Isolates of 5 different ACN3500 progeny strains)

Run breseq:
3 colonies each of 3560, 3575, 3577, 3578, and 3579 against ACN3500_NSS.gbk ref genome

In [52]:
# Creating batch (list) of short seqsamples for this seqorder

seqsamples = [SeqSample(short, row['sample_name']) for _, row in short_manifest.iloc[9:24].iterrows()]

In [14]:
from aisynbiopipeline.workflows.reference_utils import list_reference_genomes

list_reference_genomes()

['ADP1_Neidle_CDM.gbk',
 'ACN2821_CDM.gbk',
 'ACN2586_NSS.gbk',
 'ACN3500_IRZ.gbk',
 'ACN3500_NSS.gbk']

In [ ]:
## To run breseq celery worksers, activate micromamba, then call worker script:
"""
cd ~/code/AISynbioPipeline
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.breseq_task 7
"""

In [ ]:
# 6 breseq workers were already running. I am adding a 7th, hoping that it will be designated to the HTGS8F samples (hidden from currently running samples?):

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 1
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 2
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 3
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 4
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 5
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 6
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 7

In [18]:
# Where should this code go?

# Specifies and assigns breseq parameters

from pathlib import Path

def define_breseq_params(seqsample, ref_filename, poly=True, fold_coverage=300, num_processors=4):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': ref_filename,
        'polymorphism_prediction': poly,
        'limit_fold_coverage': fold_coverage,
        'num_processors': num_processors
    }
    return breseq_params

In [19]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks

results = []

for sample in seqsamples:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_NSS.gbk'),
        queue='breseq'
    )
    results.append(result)

In [28]:
sum([(r.status=='SUCCESS') for r in results])

ValueError: Exception information must include the exception type

In [27]:
for i in results:
    print(i.status)
# for i in results:
#     print(i.result) ## Check version_name to make sure it's different from previous version name

SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS


ValueError: Exception information must include the exception type

In [31]:
from aisynbiopipeline.workflows.breseq import Breseq

breseq_objects = []

for s in seqsamples:
    breseq_folder = s.library.path / 'breseq' / s.sample_name/ 'breseq_df1972644b'
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

In [34]:
for b in breseq_objects:
    print(b.title, b.exists)   

ANLstock.ACN3560.colony1 True
ANLstock.ACN3560.colony2 True
ANLstock.ACN3560.colony3 True
ANLstock.ACN3575.colony1 True
ANLstock.ACN3575.colony2 True
ANLstock.ACN3575.colony3 True
ANLstock.ACN3577.colony1 True
ANLstock.ACN3577.colony2 False
ANLstock.ACN3577.colony3 True
ANLstock.ACN3578.colony1 True
ANLstock.ACN3578.colony2 True
ANLstock.ACN3578.colony3 True
ANLstock.ACN3579.colony1 True
ANLstock.ACN3579.colony2 True
ANLstock.ACN3579.colony3 True


In [35]:
failed_seqsamples = []
for b in breseq_objects:
    if not os.path.exists(b.gd_file):
        failed_name = b.title
        print(failed_name)
        failed_seqsamples.append(SeqSample(short, failed_name))

ANLstock.ACN3577.colony2


In [37]:
# Re-running failed seqsamples

import shutil
from celery import Celery
import os

for f in failed_seqsamples:
    shutil.rmtree(f.breseq / 'breseq_df1972644b')

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks

results = []

for sample in failed_seqsamples:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_NSS.gbk'),
        queue='breseq'
    )
    results.append(result)

In [39]:
for i in results:
    print(i.status)

SUCCESS


### 3500 samples
- Create summary csv (including ver cassette coverage) and html comparison.
- Write files to 3500 breseq analysis subdir (breseq_analysis_ACN3500_NSS/).
- Create subfolder with symlinks to breseq output.

In [40]:
from aisynbiopipeline.workflows.breseq import Breseq

def create_breseq_summary(seqsample_batch, version_name, output_path, regions=None):
    
    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)
    
    rows = []
    
    for b in breseq_objects:
        
        row = {}
        
        try:
            b.count_reads()
            b.count_mutations()
            b.avg_coverage
            if regions:
                for key, value in regions.items():
                    b.get_region_average_coverage(value)
        except Exception as e:
            row.update({'seqsample': getattr(b, 'title', None)})
            # row.update(parse_seqsample_name(getattr(b, 'title', None)))
            row.update({'error': str(e),
                        'input_read_count': None,
                        'used_read_count': None,
                        'mapped_read_count': None,
                        'consensus_mutation_count': None,
                        'polymorphism_mutation_count': None,
                        'average_cov': None,}
                        )
            if regions:
                for key, value in regions.items():
                    row.update({key: None})
            rows.append(row)
            print(f"Error loading Breseq from {b.title}: {e}")
            continue
    
        row['seqsample'] = getattr(b, 'title', None)
        # row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row['error'] = None
        row['input_read_count'] = getattr(b, 'input_read_count', None)
        row['used_read_count'] = getattr(b, 'used_read_count', None)
        row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
        row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
        row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)
        row['average_cov'] = getattr(b, 'avg_coverage', None)
        if regions:
            for key, value in regions.items():
                row[key] = getattr(b, 'get_region_average_coverage', None)(value)
        rows.append(row)
    
    breseq_summary = pd.DataFrame(rows)
    
    if regions:
        for key, value in regions.items():
            breseq_summary[key+'_CN'] = breseq_summary[key]/breseq_summary['average_cov']

    # Write breseq run summary to csv
    breseq_summary.to_csv(output_path)
    
    return breseq_summary

In [41]:
def create_html_comparison(seqsample_batch, version_name, output_path):

    from aisynbiopipeline.workflows.breseq import compare_gdiff

    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)

    reference = b.params.reference
    gdiffs = [b.gd_file for b in breseq_objects]

    table_format = 'html'
    html = compare_gdiff(reference, output_path, gdiffs, format=table_format)

    return html

In [48]:
from aisynbiopipeline.workflows.reference_utils import genomic_region_from_features

def get_region_parameter(genbank_file, feature_first, feature_last):
    genome, start, stop = genomic_region_from_features(genbank_file, feature_first, feature_last)
    region = genome + ":" + str(start) + "-" + str(stop)
    return region

In [42]:
breseq_ACN3500_dir = home_dir + '/breseq_analysis_ACN3500/breseq_analysis_ACN3500_NSS/'
os.makedirs(breseq_ACN3500_dir)

In [44]:
from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path

genome3500_NSS = os.path.join(get_ref_genomes_path(), 'ACN3500_NSS.gbk')

In [58]:
# These strains are supposed to have the 6007 bp promoter deletion, including verR and parts of verB.
# Therefore, defining 'ver cassette' as verA through omega KmR cassette.

regions = {
    'ver_cassette': get_region_parameter(genome3500_NSS, 'omega KmR cassette', 'verA')
}

regions

{'ver_cassette': 'ACN3500_NSS:941311-948914'}

In [59]:
regions_to_include = ['ver_cassette']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

In [60]:
create_breseq_summary(seqsamples, 'breseq_df1972644b', os.path.join(breseq_ACN3500_dir, 'mutation_summary_ACN3500_NSS.csv'), regions_sub)
create_html_comparison(seqsamples, 'breseq_df1972644b', os.path.join(breseq_ACN3500_dir, 'mutation_comparison_ACN3500_NSS.html'))

breseq 0.39.0     http://barricklab.org/breseq

Active Developers: Barrick JE, Deatherage DE
Contact:           <jeffrey.e.barrick@gmail.com>

breseq is free software; you can redistribute it and/or modify it under the
terms the GNU General Public License as published by the Free Software 
Foundation; either version 2, or (at your option) any later version.

Copyright (c) 2008-2010 Michigan State University
Copyright (c) 2011-2022 The University of Texas at Austin

If you use breseq in your research, please cite:

  Deatherage, D.E., Barrick, J.E. (2014) Identification of mutations
  in laboratory-evolved microbes from next-generation sequencing
  data using breseq. Methods Mol. Biol. 1151: 165–188.

If you use structural variation (junction) predictions, please cite:

  Barrick, J.E., Colburn, G., Deatherage D.E., Traverse, C.C.,
  Strand, M.D., Borges, J.J., Knoester, D.B., Reba, A., Meyer, A.G. 
  (2014) Identifying structural variation in haploid microbial genomes 
  from short-rea

'/storage/nspahr/lib_analysis/Plasmidsaurus_9-13-2025_HTGS8F/breseq_analysis_ACN3500/breseq_analysis_ACN3500_NSS/mutation_comparison_ACN3500_NSS.html'

In [57]:
# Symlinks to library breseq folder
output_symlinks_dir = os.path.join(breseq_ACN3500_dir, 'symlink_to_library_breseq_folder')
os.makedirs(output_symlinks_dir, exist_ok=True)

path_to_folder = short.path / 'breseq'
dst = os.path.join(output_symlinks_dir, 'breseq')
os.symlink(path_to_folder, dst)